In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm=ChatGoogleGenerativeAI(model='gemini-2.0-flash-lite')

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'genearte a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 20.029884682s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemi

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a pizza the action!', 'explanation': 'This joke is a play on words, using a pun involving the word "piece" (as in a piece of pizza) and the word "peace" (as in joining in on the fun at a party). The humor comes from the double meaning of the word "piece" and the absurd idea of a pizza wanting to be a part of the action at a party. It\'s a lighthearted and silly joke that plays on the idea of a pizza having desires and motivations just like a person would.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b89dd-e8b0-66d6-8002-465423f73a4f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T10:14:23.151483+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b89dd-cfe4-6692-8001-9ac751d8d4a3'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a pizza the action!', 'explanation': 'This joke is a play on words, using a pun involving the word "piece" (as in a piece of pizza) and the word "peace" (as in joining in on the fun at a party). The humor comes from the double meaning of the word "piece" and the absurd idea of a pizza wanting to be a part of the action at a party. It\'s a lighthearted and silly joke that plays on the idea of a pizza having desires and motivations just like a person would.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b89dd-e8b0-66d6-8002-465423f73a4f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-11-03T10:14:23.151483+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0b89dd-cfe4-6692-8001-9ac751d8d4a3'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'p